In [1]:
import torch
from torch import nn
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:


n_samples = 1000
x, y = make_circles(n_samples, noise=0.03, random_state=42)
x_train,x_test,y_train,y_test = train_test_split(x,y,
                                                 test_size=0.2,
                                                 random_state=42)


In [4]:
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

In [5]:
from sklearn.preprocessing import StandardScaler
import torch

scaler = StandardScaler()
x_train_np = x_train.cpu().numpy()
x_test_np = x_test.cpu().numpy()

x_train_scaled = scaler.fit_transform(x_train_np)
x_test_scaled = scaler.transform(x_test_np)

x_train = torch.tensor(x_train_scaled, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test_scaled, dtype=torch.float32).to(device)
y_train = y_train.to(device).float()
y_test = y_test.to(device).float()


In [6]:
class CircleModelV1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 32)
        self.layer2 = nn.Linear(32, 32)
        self.layer3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = self.layer3(x)   # raw logit
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CircleModelV1().to(device)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

torch.cuda.manual_seed(42)
epochs = 1000
for epoch in range(epochs):
    model.train()
    y_logits = model(x_train).squeeze()
    loss = loss_function(y_logits, y_train)
    y_pred = torch.round(torch.sigmoid(y_logits))
    acc = (y_pred == y_train).float().mean() * 100

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        model.eval()
        with torch.inference_mode():
            test_logits = model(x_test).squeeze()
            test_loss = loss_function(test_logits, y_test)
            test_pred = torch.round(torch.sigmoid(test_logits))
            test_acc = (test_pred == y_test).float().mean() * 100

            print(f"epoch:{epoch}|loss:{loss:.5f}|acc:{acc:.2f}% | "
                  f"test loss:{test_loss:.5f} | test_acc:{test_acc:.2f}%")


epoch:0|loss:0.69710|acc:48.88% | test loss:0.69589 | test_acc:49.50%
epoch:100|loss:0.60110|acc:74.75% | test loss:0.61520 | test_acc:70.50%
epoch:200|loss:0.24394|acc:100.00% | test loss:0.27153 | test_acc:99.50%
epoch:300|loss:0.06580|acc:100.00% | test loss:0.09321 | test_acc:100.00%
epoch:400|loss:0.02822|acc:100.00% | test loss:0.04985 | test_acc:100.00%
epoch:500|loss:0.01632|acc:100.00% | test loss:0.03377 | test_acc:100.00%
epoch:600|loss:0.01074|acc:100.00% | test loss:0.02534 | test_acc:100.00%
epoch:700|loss:0.00763|acc:100.00% | test loss:0.02029 | test_acc:100.00%
epoch:800|loss:0.00571|acc:100.00% | test loss:0.01712 | test_acc:100.00%
epoch:900|loss:0.00444|acc:100.00% | test loss:0.01482 | test_acc:100.00%
